# GOLANG RAG TUTOR



In [16]:
import os
import glob
from dotenv import load_dotenv
from langchain_openai import OpenAIEmbeddings
from langchain_openai import ChatOpenAI
from langchain_chroma import Chroma
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.messages import SystemMessage, HumanMessage
import gradio as gr

In [10]:
MODEL = "gpt-4.1-nano"
db_name = "golang_knowledge_db"
load_dotenv(override=True)
openrouter_api_key = os.getenv('OPENROUTER_API_KEY')
if openrouter_api_key:
    print(f"Open Router API Key exists and begins {openrouter_api_key[:3]}")
else:
    print("OpenRouter API Key not set")

Open Router API Key exists and begins sk-


In [11]:
knowledge_base_path = "knowledge_docs/*.md"
files = glob.glob(knowledge_base_path, recursive=True)
print(f"Found {len(files)} files in the knowledge docs")

entire_knowledge_docs = ""

for file_path in files:
    with open(file_path, 'r', encoding='utf-8') as f:
        entire_knowledge_docs += f.read()
        entire_knowledge_docs += "\n\n"


Found 3 files in the knowledge docs


In [12]:
loader = DirectoryLoader("knowledge_docs", glob="**/*.md", loader_cls=TextLoader, loader_kwargs={'encoding': 'utf-8'})
documents = loader.load()
for doc in documents:
    doc.metadata["doc_type"] = os.path.splitext(os.path.basename(doc.metadata.get("source", "")))[0]

In [13]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = text_splitter.split_documents(documents)


embeddings = OpenAIEmbeddings(
    model="openai/text-embedding-3-large",
    base_url="https://openrouter.ai/api/v1",
    api_key=openrouter_api_key,
)

if os.path.exists(db_name):
    Chroma(persist_directory=db_name, embedding_function=embeddings).delete_collection()

vectorstore = Chroma.from_documents(documents=chunks, embedding=embeddings, persist_directory=db_name)
print(f"Vectorstore created with {vectorstore._collection.count()} documents")

Vectorstore created with 24 documents


In [14]:
SYSTEM_PROMPT_TEMPLATE = """
You are a knowledgeable, friendly tutor helping learners master fundamental Go (Golang) concepts.
You explain clearly with examples, encourage good practices, and adapt to the learner's level.
When answering, use the given context from the knowledge base. Cite or reference it when relevant.
If the context doesn't contain the answer, say so and offer to clarify or point to other resources.
Context:
{context}
"""

In [15]:
retriever = vectorstore.as_retriever()
llm = ChatOpenAI(temperature=0, model_name=MODEL, base_url="https://openrouter.ai/api/v1", api_key=openrouter_api_key)

In [17]:
def answer_question(question: str, history):
    docs = retriever.invoke(question)
    context = "\n\n".join(doc.page_content for doc in docs)
    system_prompt = SYSTEM_PROMPT_TEMPLATE.format(context=context)
    response = llm.invoke([SystemMessage(content=system_prompt), HumanMessage(content=question)])
    return response.content

In [ ]:
gr.ChatInterface(answer_question).launch()